# Reproducing Choi et al. (2025) from dandiset 002014

**Paper.** Choi J, Amjad U, Murray R, Shrivastav R, Teichert T, Goodell B, Olson M, Schaeffer DJ, Oluoch JK,
Schwerdt HN (2025). Aseptic, semi-sealed cranial chamber implants for chronic multi-channel neurochemical
and electrophysiological neural recording in nonhuman primates. *Journal of Neuroscience Methods* 420,
110467. [doi:10.1016/j.jneumeth.2025.110467](https://doi.org/10.1016/j.jneumeth.2025.110467). This attempt
used the full text of the author manuscript in PubMed Central (PMC13250822, CC BY).

**Dandiset.** [DANDI:002014](https://dandiarchive.org/dandiset/002014/0.261012.0921), version 0.261012.0921.
The paper cites no DANDI version; its data statement points to a Zenodo record
([doi:10.5281/zenodo.14825992](https://doi.org/10.5281/zenodo.14825992)), which the dandiset says it
includes together with the raw recordings. 0.261012.0921 is the only published version, and the draft holds
the same file (same blob) and the same metadata, so the choice changes nothing the analysis uses.

**What the dandiset holds.** One NWB file: one session (09/27/2024) of monkey T, the session of Figures 8
and 9. Monkey J, and the surgical, culture, drainage and electrode-yield results that make up most of the
paper, are not in it.

**Outside input.** The paper's trial lists in the Zenodo record (successful large and small reward trials)
are used only to check the trial selection built here from the dandiset's events.

**Code read.** The paper links no analysis code. The dandiset links its conversion code,
[catalystneuro/schwerdt-lab-to-nwb](https://github.com/catalystneuro/schwerdt-lab-to-nwb) at commit
6edf72904a64d92f3b6d9268d38a31ffab95f36e (`choi_2025` and the interfaces it uses). It sets the layout of the
trial-aligned FSCV table (60 s rows centred on the trial-start event), rebuilds the LFP from 60 s trial pieces
placed by the trial times, places the raw ephys by a starting time of 0 and a rate, and places the raw FSCV
by a single starting time.

| Claim | Figure | Tested |
|---|---|---|
| fig8a-single-trial-dopamine | 8A | yes |
| fig8b-dopamine-large-vs-small | 8B | yes |
| fig9a-spikes-task-aligned | 9A | yes |
| fig9bc-unit-waveform-isi | 9B, 9C | yes |
| fig9d-beta-suppression-large | 9D | yes |
| recordings-two-monkeys | Abstract, 3.5 | blocked: data missing (monkey J) |
| scfe-functional-yield | 3.5 | blocked: data missing |
| scfe-background-yield | 3.5 | blocked: data missing |
| mcfe-functional-yield | 3.5 | blocked: data missing |
| dopamine-signal-yield | 3.5 | blocked: data missing |
| asepsis-cultures | 3.2 | blocked: data missing |
| fig6-drainage-volume | 6 | blocked: data missing |
| fig7-bone-growth-margins | 7 | blocked: data missing |

In [ ]:
import time
import io
import warnings
import json
import urllib.request

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from scipy import io as sio
from scipy import signal, stats

import chamber

warnings.filterwarnings("ignore", category=RuntimeWarning)
pd.set_option("display.width", 200)
pd.set_option("display.max_columns", 30)
started = time.time()
f = chamber.open_file()
events = chamber.load_events(f)
trials = chamber.trial_table(f, events)
fscv = chamber.load_fscv_table(f)
lfp_t, lfp_v = chamber.load_lfp(f)
thr = chamber.spike_units(f, "processing/ecephys/thresholded_units")
plx = chamber.spike_units(f, "units")
print(f"dandiset {chamber.DANDISET_ID} version {chamber.DANDISET_VERSION}, {chamber.ASSET_PATH}; "
      f"loaded in {time.time() - started:.0f} s")

## Data check

Each check of the metadata against the values is stated under its output.

In [ ]:
def text(path):
    x = f[path][()]
    return x.decode() if isinstance(x, bytes) else x

print("Subject:", {k: text(f"general/subject/{k}") for k in ("subject_id", "species", "sex", "age")})
print("Subject description:", text("general/subject/description"))
print("Session start:", text("session_start_time"), "; session id:", text("general/session_id"))
print("Experiment description:", text("general/experiment_description")[:160], "...")
print("Keywords:", [k.decode() for k in f["general/keywords"][()]])
print("Related publication:", [k.decode() for k in f["general/related_publications"][()]])

el = f["general/extracellular_ephys/electrodes"]
cols = ["location", "channel_name", "group_name", "recording_opened", "recording_closed", "DspLowCutFrequency", "DspHighCutFrequency", "InputRange"]
print("\nElectrodes table:")
print(pd.DataFrame({c: [x.decode() if isinstance(x, bytes) else x for x in el[c][()]] for c in cols}).to_string())

series = {
    "raw ephys": "acquisition/electrical_series",
    "bipolar LFP": "processing/ecephys/FilteredEphys/bipolar_lfp_series",
    "raw FSCV current": "acquisition/fscv_response_series",
    "FSCV applied voltage": "stimulus/presentation/fscv_excitation_series",
}
rows = []
for name, path in series.items():
    g = f[path]
    if "timestamps" in g:
        ts = g["timestamps"]
        span = (ts[0], ts[-1])
        step = np.median(np.diff(ts[:5000]))
    else:
        rate = g["starting_time"].attrs["rate"]
        span = (g["starting_time"][()], g["starting_time"][()] + g["data"].shape[0] / rate)
        step = 1 / rate
    rows.append({"series": name, "shape": g["data"].shape, "dtype": str(g["data"].dtype),
                 "unit": g["data"].attrs["unit"], "conversion": g["data"].attrs["conversion"],
                 "electrodes": f[path + "/electrodes"][()].tolist() if "electrodes" in f[path] else None, "first_s": round(span[0], 3),
                 "last_s": round(span[1], 3), "sample_step_s": step})
print("\nTime series (times in s from session start):")
print(pd.DataFrame(rows).to_string(index=False))

One NWB file, one session (09/27/2024) of monkey T, the session of Figures 8 and 9. Monkey J is not
in the dandiset. The subject description ("Two adult female rhesus monkeys were used for in vivo
validation of the s-µIP device"), the session and experiment descriptions and the keyword
"micro-invasive neural implant" describe micro-invasive probes, the subject of another paper by the
lab (the conversion code's `amjad_2025` project); they do not describe this paper's chamber. The
related publication is this paper. The electrode sites (c7b, c5c, c5d for ephys, c8ds for FSCV) are
those of Figures 8 and 9.

The raw ephys has no timestamps: it starts at 0 s, at 32 kHz, so it covers 0 to about 18336 s from the
session start. The LFP, the FSCV series, the trials and the events cover about 11825 to 31535 s. The
two ranges overlap only in part; the clock checks below deal with this.

In [ ]:
counts = chamber.raw_event_counts(f)
print("Events as stored and after removing repeats:")
print(counts.to_string(index=False))
print("\nTrials:", len(trials), "; trial windows", np.unique(np.round(trials.stop_time - trials.start_time, 6)), "s;",
      "trial-start events matching the trials' midpoint_time:",
      int(np.sum([np.min(np.abs(events["intended trial start"] - m)) < 1e-6 for m in trials.trial_start])))
tags = f["intervals/trials/tags"][()]
print("Trial tags:", pd.Series([t.decode() for t in tags]).value_counts().to_dict())
print("Trials with more than one V:", int((trials.n_V > 1).sum()))
print("\nTrial outcomes (first error event, or the reward size):")
print(pd.crosstab(trials.error.fillna("none"), trials.reward.fillna("no reward")).to_string())
succ = trials[trials.success]
print("\nSuccessful trials: large", int((succ.reward == "large").sum()), ", small", int((succ.reward == "small").sum()))
print("Successful trials by side and reward:")
print(pd.crosstab(succ.side, succ.reward).to_string())
lat = pd.DataFrame({"V - trial start": succ.V - succ.trial_start, "RW - V": succ.RW - succ.V})
print("\nLatencies in successful trials (s):")
print(lat.describe(percentiles=[0.05, 0.5, 0.95]).round(3).to_string())

In [ ]:
# The paper's trial lists on Zenodo (doi:10.5281/zenodo.14825992), 1-based trial ids.
ZENODO = "https://zenodo.org/api/records/14825992/files/"
lists = {}
for size, name in (("large", "big"), ("small", "small")):
    url = ZENODO + urllib.request.quote(f"09272024_nlx_c8c_value object start_{name} rew and success_tr1-1411.mat") + "/content"
    with urllib.request.urlopen(url) as r:
        lists[size] = set(sio.loadmat(io.BytesIO(r.read()))["trids_out"].ravel().tolist())
for size in ("large", "small"):
    mine = set(succ.trial[succ.reward == size])
    print(f"{size}: Zenodo list {len(lists[size])} trials, dandiset events {len(mine)}, both {len(mine & lists[size])}, "
          f"only in the dandiset {sorted(mine - lists[size])}, only in the list {sorted(lists[size] - mine)}")
print("Last trial:", trials.iloc[-1][["trial", "reward", "error", "trial_start", "stop_time"]].to_dict(),
      "; last LFP sample", round(lfp_t[-1], 3))

The event table stores each event once per trial window that contains it (60 s windows overlap), so
labels hold about five copies of each event (7221 stored trial-start events for 1412 distinct ones) and
are not sorted; repeats are removed here. After that there are 1412
trial-start events for 1411 trials, and every trial's `midpoint_time` is one of them. The trial tags are
all "initial cue display" and carry no trial type; reward size and outcome come from the events. Every
trial has at most one V.

Built from the events, the successful trials match the paper's Zenodo lists exactly (499 small; 451
large) except trial 1411, the last, which the events give as a successful large trial and the list
leaves out. Its window runs to the end of the LFP. Trial 1411 is kept here, as the dandiset's events
mark it successful; the difference is one trial in 452.

The paper describes the reward size as depending on whether the target is at the top or bottom of the
screen; the events code left and right conditions, and both sizes occur on both sides. The dandiset's
description also speaks of left and right targets. The data do not say which is right.

In [ ]:
# Units and ranges of every variable used.
print("Trial-aligned FSCV table: rows", fscv["da"].shape[0], ", samples per row", fscv["da"].shape[1],
      ", rate", np.unique(fscv["rate"]), "Hz, good", int(fscv["good"].sum()), "of", len(fscv["good"]))
print("Rows start at the trials' start_time:", bool(np.allclose(fscv["start_time"], trials.start_time)),
      "; row time axis", fscv["t_rel_start"][[0, -1]], "s from the trial start")
for k in ("da", "ph", "m", "iox"):
    x = fscv[k]
    print(f"  {k:3s}: unit not stated; range {np.nanmin(x):.3f} to {np.nanmax(x):.3f}; NaN {np.isnan(x).mean():.3%} of samples, "
          f"in {np.isnan(x).any(1).sum()} rows")
print("Columns holding NaN in more than 5% of rows (s from trial start):",
      np.round(fscv["t_rel_start"][np.isnan(fscv["da"]).mean(0) > 0.05][[0, -1]], 1))
print("Dopamine NaN in successful trials within V-1 s to V+6 s:",
      f"{np.isnan(chamber.dopamine_aligned(fscv, succ, np.arange(-1, 6.01, 0.1))[1]).mean():.2%} of samples")

ok = ~np.isnan(lfp_v)
print(f"\nBipolar LFP: values {np.nanmin(lfp_v):.2e} to {np.nanmax(lfp_v):.2e} as stored, SD {np.nanstd(lfp_v):.2e};"
      f" NaN {1 - ok.mean():.2%}; sample step {np.median(np.diff(lfp_t[:100000])):.6f} s; gaps over 1.5 ms: "
      f"{int(np.sum(np.diff(lfp_t) > 0.0015))}")
print("The stored conversion factor would scale them to", f"{np.nanstd(lfp_v) * f['processing/ecephys/FilteredEphys/bipolar_lfp_series/data'].attrs['conversion']:.1e}", "V SD")
seg_raw = chamber.raw_segment(f, 3000, 300, slice(0, 3))
raw_bip = signal.decimate(signal.decimate(seg_raw[:, 0] - seg_raw[:, 1], 8), 4)
k0 = np.searchsorted(lfp_t, 20000)
fr, p_lfp = signal.welch(np.nan_to_num(lfp_v[k0:k0 + 300000]), fs=1000, nperseg=4096)
_, p_raw = signal.welch(raw_bip, fs=1000, nperseg=4096)
bands = [(1, 4), (4, 8), (8, 13), (13, 30), (30, 58), (58, 62), (62, 100), (100, 300)]
print("\nPower spectral density (V^2/Hz): stored LFP as stored, and raw c7b minus c5c in volts (300 s each)")
print(pd.DataFrame([{"band_Hz": f"{a}-{b}", "stored_LFP": p_lfp[(fr >= a) & (fr < b)].mean(),
                     "raw_bipolar": p_raw[(fr >= a) & (fr < b)].mean()} for a, b in bands]).to_string(index=False))
print("\nRaw ephys in volts (300 s from 3000 s): range per channel", np.round(seg_raw.min(0) * 1e6), "to", np.round(seg_raw.max(0) * 1e6), "uV")
full_scale = f["acquisition/electrical_series/data"].attrs["conversion"] * 32767
clip = []
for start in np.linspace(100, 18000, 6):
    x = chamber.raw_segment(f, start, 30, slice(0, 3))
    clip.append(np.mean(np.abs(x) >= 0.999 * full_scale, axis=0))
print("Fraction of samples at the input range (+-1 mV), 30 s at six points, per channel (c7b, c5c, c5d):", np.round(np.array(clip), 4).tolist())
peak10 = p_raw[(fr >= 9.5) & (fr <= 10.5)].max() / np.median(p_raw[(fr >= 4) & (fr <= 8)])
print(f"Raw bipolar: peak density at 9.5-10.5 Hz is {peak10:.0f} times the median at 4-8 Hz; stored LFP: "
      f"{p_lfp[(fr >= 9.5) & (fr <= 10.5)].max() / np.median(p_lfp[(fr >= 4) & (fr <= 8)]):.1f} times")

The trial-aligned FSCV table has a row for every trial, all marked good, each 601 samples at 10 Hz from
30 s before to 30 s after the trial start. Its columns state no units. The dopamine values (about -54 to
42) fit nM, the unit of Δ[DA] in the lab's earlier work, but the file does not say so. NaN gaps occur in
most rows.

The table "bipolar_lfp_series" is described as beta-band LFP. Its spectrum is broadband, falling from
1 Hz up, like the raw c7b minus c5c difference, with the line-noise band (58 to 62 Hz) removed (the
Zenodo record speaks of spectral interpolation). Its values are already in volts (at 1 to 4 Hz its
density matches the raw difference in volts), but the series carries the raw int16 conversion factor
(3.05e-8), so a reader that applies it gets values about 3e7 times too small. Beta power is computed
here from the stored values with a band-pass filter, as set in the claims file.

In [ ]:
# Spikes: the thresholded table (the paper's Zenodo file) and the Plexon unit.
g = f["processing/ecephys/thresholded_units"]
print("Thresholded table description:", g.attrs["description"])
print("Electrodes of its units:", g["electrodes"][()].tolist())
for name, st in thr.items():
    print(f"  thresholded unit {name}: {len(st)} spikes, {st.min():.3f} to {st.max():.3f} s, sorted {bool(np.all(np.diff(st) >= 0))}")
for name, st in plx.items():
    print(f"  Plexon unit {name}: {len(st)} spikes, {st.min():.3f} to {st.max():.3f} s, "
          f"sorted {bool(np.all(np.diff(st) >= 0))}, decreasing steps {int(np.sum(np.diff(st) < 0))}")
main_unit = max(thr, key=lambda k: len(thr[k]))
spk = thr[main_unit]
plx_spk = np.sort(next(iter(plx.values())))
nearest = np.minimum(np.abs(spk[np.clip(np.searchsorted(spk, plx_spk), 0, len(spk) - 1)] - plx_spk),
                     np.abs(spk[np.clip(np.searchsorted(spk, plx_spk) - 1, 0, len(spk) - 1)] - plx_spk))
print(f"Plexon spikes within 0.2 ms of a spike of thresholded unit {main_unit}: {np.sum(nearest < 2e-4)} of {len(plx_spk)}; "
      f"within 1 us: {np.sum(nearest < 1e-6)}")
wf = chamber.thresholded_waveforms(f, list(thr).index(main_unit), max_spikes=20000)
print(f"Waveforms: {wf.shape[1]} samples; unit attribute '{g['waveforms'].attrs['unit']}'; mean waveform {wf.mean(0).min():.1f} to "
      f"{wf.mean(0).max():.1f} as stored; fraction of spikes whose largest |value| exceeds 40: {np.mean(np.abs(wf).max(1) > 40):.3f}")
es_dur = f["acquisition/electrical_series/data"].shape[0] / chamber.RAW_RATE
print(f"Raw ephys duration {es_dur:.2f} s; spikes {spk.min():.3f} to {spk.max():.3f} s")

The thresholded table holds unit "0" with 3 spikes and unit "1" with the rest; the description says
-1 marks poor-quality spikes, and no -1 occurs. Unit 1 is used below. The Plexon unit is a different
sorting of the same channel: part of its spikes coincide with unit 1's (counts above), and its spike
times are stored out of order. The waveforms are stated in volts, but their values (a mean trough
of about -53 over the first 20000 spikes) are microvolts: every spike exceeds 40 in absolute value, which fits the 40 µV threshold
in the name of the paper's Zenodo file (`csc23_100_40uv.mat`). Spike times run from 0 to the end of
the raw ephys, which suggests the raw ephys clock; the next cell checks this.

In [ ]:
# Clock 1: where the raw ephys falls on the trial clock. Segments of raw c7b minus c5c are located in the
# stored LFP, which is placed on the trial clock by the trials' start times.
candidate, clock = chamber.locate_raw_in_lfp(f, lfp_t, lfp_v)
print(f"Raw c7b minus c5c at {candidate['raw_time']:.0f}-{candidate['raw_time'] + chamber.CLOCK_SEGMENT_S:.0f} s, searched in the whole stored LFP: "
      f"best at offset {candidate['offset']:.4f} s, r = {candidate['r']:.4f}")
print("Stored LFP segments searched in the raw signal within 60 s of that offset:")
print(clock.round(4).to_string(index=False))
OFFSET = float(np.median(clock.offset))
print(f"\nOffset of the trial clock from the raw ephys clock: median {OFFSET:.4f} s; range {clock.offset.min():.4f} to "
      f"{clock.offset.max():.4f} s; correlation {clock.r.min():.3f} to {clock.r.max():.3f}, best 1 s or more away at most {clock.best_r_1s_away.max():.3f}")
drift = np.polyfit(clock.raw_time, clock.offset, 1)[0]
print(f"Drift {drift * 3600 * 1000:.2f} ms per hour")

# Clock 2: the spike times against raw c5d (300-6000 Hz), spike-triggered, over 300 s.
lag_ms, snip = chamber.spike_triggered_raw(f, spk, 6000.0, 300.0, channel=2)
sta_c5d = snip.mean(0)
k_min = int(np.argmin(sta_c5d))
per_spike = lag_ms[np.argmin(snip, 1)]
print(f"\nSpike-triggered average of raw c5d over {len(snip)} spikes: trough {sta_c5d[k_min] * 1e6:.1f} uV at {lag_ms[k_min]:.2f} ms "
      f"from the stored spike time; peak-to-peak {np.ptp(sta_c5d) * 1e6:.1f} uV; mean |signal| far from the spikes "
      f"{np.median(np.abs(snip)) * 1e6:.1f} uV")
print("Per-spike trough lag (ms), 10th/50th/90th percentile:", np.round(np.percentile(per_spike, [10, 50, 90]), 2))
rng_clock = np.random.default_rng(0)
null_lag, null_snip = chamber.spike_triggered_raw(f, spk + rng_clock.uniform(0.05, 0.5, len(spk)), 6000.0, 300.0, channel=2)
print(f"Spike times moved by a random 50-500 ms: trough {null_snip.mean(0).min() * 1e6:.1f} uV")
for ch, site in ((0, "c7b"), (1, "c5c")):
    _, sn = chamber.spike_triggered_raw(f, spk, 6000.0, 300.0, channel=ch)
    print(f"Same at raw {site}: trough {sn.mean(0).min() * 1e6:.1f} uV, peak-to-peak {np.ptp(sn.mean(0)) * 1e6:.1f} uV")
SPIKE_LAG = lag_ms[k_min] / 1000
print("Mean stored waveform trough at sample", int(np.argmin(wf.mean(0))), f"({np.argmin(wf.mean(0)) / 32:.2f} ms after the waveform start)")

# Absolute times.
opened = pd.Timestamp(text("session_start_time"))
closed_hdr = el["recording_closed"][()][0].decode()
raw_end_trial_clock = OFFSET + es_dur
print(f"\nSession start {opened}; Neuralynx header: recording closed {closed_hdr}")
print(f"The raw ephys on the trial clock runs from {opened + pd.Timedelta(seconds=OFFSET)} to {opened + pd.Timedelta(seconds=raw_end_trial_clock)}")
print(f"Header open-to-close span {(pd.Timestamp(closed_hdr) - pd.Timestamp(opened.tz_localize(None))).total_seconds():.0f} s; raw samples {es_dur:.0f} s")

The raw ephys is found in the stored LFP at a constant offset: raw time 0 is 11812.1 s on the trial
clock. The first search covers the whole LFP; the eight checks across the session each find the
same offset, to within a few ms that grow steadily with time (the drift printed above), with
correlations near 1 against at most about 0.35 elsewhere in the window. The raw search was restricted
to 60 s around the candidate at those points because raw c5c sits at the ±1 mV input range in about
1.7% of samples and the raw difference carries a large 10 Hz artefact (the FSCV scan rate), which the
stored LFP does not. Raw c5d shows a spike at every stored spike time, but about 7 ms before it: the
spike-triggered trough is at -7.1 ms, and nearly every spike's trough lies between -7.4 and -6.9 ms.
Moving the spike times by 50 to 500 ms removes it, and the other two sites do not show it. So the spike
times are on the raw ephys clock, 7 ms late. The 7 ms is small next to the 100 ms bins used below and is
left as stored. The raw ephys and the spikes therefore do not
share the clock of the trials, events, LFP and FSCV table: as stored they are 11812.1 s early. The file
puts the raw ephys at 0 s (it stores a starting time and a rate, no timestamps), and its spike times
are counted from the start of the raw samples.

The absolute times do not fit together either. The Neuralynx header gives the recording as opened at
the session start (08:49:52) and closed at 15:12:24, a span of about 22950 s, but there are 18336 s of
samples. On the trial clock the raw samples end at 17:12:20, two hours after the header's closing time.
The data cannot say which time is wrong: a gap of about 1 h 17 min at the start of the raw recording
that the file drops, and a two-hour error in the trial times (a time zone, for example), would fit.
Relative times within each clock are consistent, which is what the analysis needs.

From here on, spike times are placed on the trial clock by adding the measured offset. The next cell
checks this against the task.

In [ ]:
# Contrasts: does each stream respond to the task at V on the trial clock?
rng_check = np.random.default_rng(1)
V = succ.V.to_numpy()
spk_trial = spk + OFFSET
def null_alignments(n, rng):
    """V replaced by a uniform time within each successful trial's span (its start to the next trial's start)."""
    return [rng.uniform(succ.trial_start.to_numpy(), succ.trial_end.to_numpy()) for _ in range(n)]
nulls = null_alignments(100, rng_check)
null_mod = np.array([chamber.modulation(chamber.psth(spk_trial, a)[1]) for a in nulls])
rows = []
for label, extra in [("measured offset", 0.0), ("offset - 5 s", -5.0), ("offset + 5 s", 5.0)]:
    _, c = chamber.psth(spk_trial + extra, V)
    rows.append({"spike clock": label, "modulation": chamber.modulation(c)})
_, c0 = chamber.psth(spk, V)
rows.append({"spike clock": "as stored (no offset)", "modulation": chamber.modulation(c0),
             })
tab = pd.DataFrame(rows)
tab["null 95th pct"] = np.percentile(null_mod, 95)
print("Spikes: variance across 100 ms bins of the trial-averaged rate from V-1 s to V+3 s (spikes/s squared)")
print(tab.round(2).to_string(index=False))
print("Successful trials with V inside the stored spike times' range:", int(np.sum((V > spk.min()) & (V < spk.max()))), "of", len(V))

# LFP: evoked potential at V against random times in the same trials.
def evoked_raw(times, pad=1.0):
    out = []
    for t0 in times:
        k = np.searchsorted(lfp_t, t0 - pad)
        seg = lfp_v[k:k + 2001]
        if len(seg) == 2001 and not np.isnan(seg).any():
            out.append(seg - seg[:1000].mean())
    return np.array(out)
ev_true = evoked_raw(V)
ev_null = [evoked_raw(a) for a in nulls[:30]]
amp = lambda e: np.ptp(e.mean(0)[1000:1500])
print(f"\nLFP: peak-to-peak of the mean LFP 0-0.5 s after V, {len(ev_true)} trials: {amp(ev_true) * 1e6:.1f} uV; "
      f"random times in the same trials, 95th percentile {np.percentile([amp(e) for e in ev_null], 95) * 1e6:.1f} uV")

# Dopamine: when do large and small trials diverge, relative to V?
t_da, da_al = chamber.dopamine_aligned(fscv, succ)
big = (succ.reward == "large").to_numpy()
tt = stats.ttest_ind(da_al[big], da_al[~big], nan_policy="omit").statistic
tt = np.asarray(tt)
run = np.convolve(np.abs(tt) > 3, np.ones(5, int), "valid") == 5
first = t_da[np.argmax(run)] if run.any() else None
print(f"\nDopamine: first time from V at which |t| (large against small) exceeds 3 for 5 samples running: {first:.1f} s;"
      f" largest |t| before V: {np.nanmax(np.abs(tt[t_da < 0])):.2f}")

With the measured offset the spike rate is locked to V: the variance of the trial-averaged rate across
bins is far above that at random times in the same trials. Moved 5 s later it is not; moved 5 s earlier
it is still above chance, as the window then holds the time of the reward, about 4.8 s after V. As
stored it is not. As stored, V falls within the range of the spike times in only 399 of 951 successful trials. This
contrast places the spikes on the task to within a few seconds (the response is broad, so smaller shifts
keep a high variance); the millisecond alignment rests on the two steps above.

The LFP shows an evoked potential after V larger than at random times in the same trials, and the
dopamine of large and small trials diverges only after V (from 2.5 s), with no run of |t| above 3
before it (largest 2.04). Both are
consistent with the LFP and the FSCV table sharing the trial clock, as their construction from the
trials' start times implies.

In [ ]:
# Raw FSCV: can the trial-aligned oxidation current be found in it?
rs = f["acquisition/fscv_response_series"]
ex = f["stimulus/presentation/fscv_excitation_series"]
n_scan = rs["data"].shape[0] // chamber.FSCV_SAMPLES_PER_SCAN
scans = rs["data"][()].reshape(n_scan, chamber.FSCV_SAMPLES_PER_SCAN)
scan_t = rs["timestamps"][::chamber.FSCV_SAMPLES_PER_SCAN]
e0 = ex["data"][:chamber.FSCV_SAMPLES_PER_SCAN]
print(f"Raw FSCV: {n_scan} scans of {chamber.FSCV_SAMPLES_PER_SCAN} samples; scan interval {np.unique(np.round(np.diff(scan_t), 4))} s; "
      f"span {scan_t[0]:.2f} to {scan_t[-1]:.2f} s ({(scan_t[-1] - scan_t[0]) / 60:.1f} min, the trials span "
      f"{(trials.stop_time.iloc[-1] - trials.start_time.iloc[0]) / 60:.1f} min)")
print(f"Applied voltage per scan {e0.min():.3f} to {e0.max():.3f} V (paper: -0.4 to 1.3 V); "
      f"current stated in {rs['data'].attrs['unit']}, values {np.percentile(scans, 1):.0f} to {np.percentile(scans, 99):.0f} (1st to 99th percentile)")
j = int(np.argmin(np.abs(e0[:len(e0) // 2] - 0.6)))
sos = signal.butter(2, [0.02, 0.5], "bandpass", fs=10, output="sos")
cur = signal.sosfiltfilt(sos, scans[:, j - 1:j + 2].mean(1))
energy = np.cumsum(np.r_[0, cur ** 2])
rows = []
for k in np.linspace(0, len(trials) - 1, 7).astype(int):
    y = signal.sosfiltfilt(sos, fscv["iox"][k])
    r = signal.correlate(cur, y, "valid") / np.sqrt((energy[len(y):] - energy[:-len(y)]) * np.sum(y ** 2))
    i = int(np.argmax(np.abs(r)))
    rows.append({"trial": k + 1, "row start s": round(fscv["start_time"][k], 2), "best lag s": round(scan_t[i] - fscv["start_time"][k], 1),
                 "best r": round(r[i], 3), "30th best |r|": round(np.sort(np.abs(r))[-30], 3)})
print(f"\nThe 'iox' row of a trial searched in the raw current at {e0[j]:.2f} V (both 0.02-0.5 Hz):")
print(pd.DataFrame(rows).to_string(index=False))
del scans

The raw FSCV holds 196800 scans placed back to back at 10 Hz from the first trial's start, 328 min in
all, longer than the 302 min of trials, so the minute files were joined without their gaps; the
conversion sets only a starting time. The current is stated in amperes but its values (around 1000)
are nA. The trial-aligned oxidation current cannot be found in it: the best lags differ from trial to
trial and their correlations are no better than the 30th best. The raw FSCV's clock could therefore
not be established. No claim uses the raw FSCV.

No recording is deposited twice. The thresholded spikes and the Plexon unit are two sortings of the
same channel; the analysis uses the thresholded file, which is the one in the paper's data record.

## fig8a-single-trial-dopamine

> In a single trial (trial 133 of the session), dopamine concentration changes at site c8ds fluctuate,
> especially after the value cue display.

Criterion: the SD of dopamine (referenced at V) over 0 to 5 s after V against the 5 s before V, ratio
above 1, for trial id 133 (1-based) or index 133.

In [ ]:
t_full, da_full = chamber.dopamine_aligned(fscv, trials)
def sd_ratio(row):
    after = da_full[row, (t_full >= 0) & (t_full <= 5)]
    before = da_full[row, (t_full >= -5) & (t_full < 0)]
    return np.nanstd(after) / np.nanstd(before), np.isnan(after).mean(), np.isnan(before).mean()
for label, trial_id in (("trial id 133 (1-based)", 133), ("index 133 (trial id 134)", 134)):
    row = trial_id - 1
    r, na, nb = sd_ratio(row)
    tr = trials.iloc[row]
    print(f"{label}: reward {tr.reward}, error {tr.error}, V at {tr.V - tr.trial_start:.2f} s, RW at {tr.RW - tr.V:.2f} s from V; "
          f"SD ratio after/before {r:.2f} (NaN after {na:.0%}, before {nb:.0%})")
ratios = np.array([sd_ratio(i - 1)[0] for i in succ.trial])
print(f"Reference, all {len(ratios)} successful trials: median ratio {np.nanmedian(ratios):.2f}, "
      f"5th-95th percentile {np.nanpercentile(ratios, 5):.2f} to {np.nanpercentile(ratios, 95):.2f}, above 1 in {np.nanmean(ratios > 1):.0%}")
fig, ax = plt.subplots(figsize=(7, 3))
for trial_id in (133, 134):
    ax.plot(t_full, da_full[trial_id - 1], label=f"trial {trial_id}")
ax.axvline(0, color="k", lw=0.5)
ax.set(xlabel="time from V (s)", ylabel="dopamine referenced at V (stated unit: none)", xlim=(-10, 10))
ax.legend()
print("Plotted range:", np.round(np.nanmin(da_full[[132, 133]]), 2), "to", np.round(np.nanmax(da_full[[132, 133]]), 2))

**Result: reproduced.** In trial 133 (1-based, a successful large reward trial) the SD of dopamine over the
5 s after V is 2.02 times that over the 5 s before; under the other reading (index 133, trial 134) the ratio
is 2.04, with 36% and 46% of its samples NaN. The test is weak: across all 951 successful trials the median
ratio is 1.27 and 60% are above 1.

## fig8b-dopamine-large-vs-small

> Trial-averaged dopamine at site c8ds, aligned to the value cue, increases more for large than for
> small reward trials, especially between the value cue display and reward.

Criterion: per successful trial, mean dopamine (referenced at V) from V to RW; large above small, two-sided
Mann-Whitney p < 0.05.

In [ ]:
def window_by_trial(da, t_axis, lo, hi):
    """Mean of each row between per-trial bounds lo[i] and hi[i] (s from V)."""
    out = np.full(len(da), np.nan)
    for i in range(len(da)):
        m = (t_axis >= lo[i]) & (t_axis <= hi[i])
        if np.any(~np.isnan(da[i, m])):
            out[i] = np.nanmean(da[i, m])
    return out
rw = (succ.RW - succ.V).to_numpy()
main = window_by_trial(da_al, t_da, np.zeros(len(succ)), rw)
def compare(values, mask_big, label):
    a, b = values[mask_big], values[~mask_big]
    a, b = a[~np.isnan(a)], b[~np.isnan(b)]
    p = stats.mannwhitneyu(a, b, alternative="two-sided").pvalue
    d = (a.mean() - b.mean()) / np.sqrt((a.var(ddof=1) + b.var(ddof=1)) / 2)
    return {"window": label, "n_large": len(a), "n_small": len(b), "mean_large": a.mean(), "mean_small": b.mean(),
            "difference": a.mean() - b.mean(), "cohen_d": d, "mann_whitney_p": p}
rows = [compare(main, big, "V to RW (main)")]
print("Dropped (no dopamine sample in V to RW): large", int(np.isnan(main[big]).sum()), ", small", int(np.isnan(main[~big]).sum()))
rows.append(compare(window_by_trial(da_al, t_da, np.zeros(len(succ)), np.full(len(succ), 2.0)), big, "0 to 2 s"))
rows.append(compare(window_by_trial(da_al, t_da, rw, rw + 3), big, "RW to RW + 3 s"))
rows.append(compare(window_by_trial(da_al, t_da, np.full(len(succ), -3.0), np.zeros(len(succ))), big, "-3 to 0 s (before V)"))
half = (succ.trial <= len(trials) / 2).to_numpy()
for name, m in (("first half of session", half), ("second half of session", ~half)):
    r = compare(main[m], big[m], "V to RW")
    r["window"] = f"V to RW, {name}"
    rows.append(r)
print(pd.DataFrame(rows).round(4).to_string(index=False))
fig, ax = plt.subplots(figsize=(7, 3))
for m, lab, col in ((big, "large", "tab:red"), (~big, "small", "tab:blue")):
    mu = np.nanmean(da_al[m], 0)
    se = np.nanstd(da_al[m], 0) / np.sqrt(np.sum(~np.isnan(da_al[m]), 0))
    ax.plot(t_da, mu, color=col, label=lab)
    ax.fill_between(t_da, mu - se, mu + se, color=col, alpha=0.3)
ax.axvline(0, color="k", lw=0.5)
ax.axvline(np.median(rw), color="k", ls="--", lw=0.5)
ax.set(xlabel="time from V (s); dashed: median RW", ylabel="dopamine referenced at V", xlim=(-10, 10))
ax.legend()
print("Plotted means range:", np.round(np.nanmin([np.nanmean(da_al[big], 0), np.nanmean(da_al[~big], 0)]), 2), "to",
      np.round(np.nanmax([np.nanmean(da_al[big], 0), np.nanmean(da_al[~big], 0)]), 2), "; median RW", round(np.median(rw), 2), "s")

**Result: reproduced.** From V to reward, dopamine referenced at V is higher in large than in small reward
trials (means 1.87 against 1.11, difference 0.76 in the file's unstated unit, Cohen's d 0.18, Mann-Whitney
p = 0.0037; 434 and 466 trials, 18 and 33 dropped for lack of samples). The difference is small in the first
2 s after V (p = 0.49) and largest after reward (RW to RW + 3 s, difference 1.54, p = 0.0002), and the two
conditions do not differ before V (p = 0.52). It holds in the second half of the session (p = 0.0098) and
not significantly in the first (p = 0.089).

## fig9a-spikes-task-aligned

> Spikes recorded at site c5d on a chamber-implanted CFE can be shown as a raster and a peristimulus
> time histogram aligned to the value cue display, for large and small reward trials.

Criterion: (i) at least 90% of successful trials have a spike within V-1 s to V+3 s; (ii) the variance
across 100 ms bins of the PSTH exceeds the 95th percentile of 200 random alignments within the same
trials. Spike times on the trial clock with the offset measured in the data check.

In [ ]:
rng_9a = np.random.default_rng(9)
def fig9a_test(spike_times, label):
    centers, counts = chamber.psth(spike_times, V)
    has_spike = (counts.sum(1) > 0).mean()
    obs = chamber.modulation(counts)
    null = np.array([chamber.modulation(chamber.psth(spike_times, a)[1]) for a in null_alignments(200, rng_9a)])
    return {"spikes": label, "trials with a spike": round(has_spike, 4), "modulation": round(obs, 2),
            "null 95th pct": round(np.percentile(null, 95), 2), "null max": round(null.max(), 2),
            "p (random alignments)": (np.sum(null >= obs) + 1) / (len(null) + 1)}, centers, counts
res_main, centers, counts = fig9a_test(spk + OFFSET, f"thresholded unit {main_unit}, measured offset (main)")
res_plx, _, counts_plx = fig9a_test(plx_spk + OFFSET, "Plexon unit, measured offset")
res_raw, _, _ = fig9a_test(spk, f"thresholded unit {main_unit}, as stored")
res_lag, _, _ = fig9a_test(spk + OFFSET + SPIKE_LAG, f"thresholded unit {main_unit}, offset and 7 ms lag")
print(pd.DataFrame([res_main, res_plx, res_raw, res_lag]).to_string(index=False))
# Variants: other events and a longer window, 250 ms bins.
rows = []
fixation = events["value cue fixation started"]
v_fix = fixation[np.searchsorted(fixation, V)]
for label, align, window in (("V, -1 to 6 s", V, (-1.0, 6.0)), ("RW, -2 to 3 s", succ.RW.to_numpy(), (-2.0, 3.0)),
                             ("value cue fixation, -2 to 3 s", v_fix, (-2.0, 3.0))):
    _, c = chamber.psth(spk + OFFSET, align, window=window, bin_s=0.25)
    # null: the same random times as for V, moved by each trial's event-to-V interval
    nm = [chamber.modulation(chamber.psth(spk + OFFSET, a + (align - V), window=window, bin_s=0.25)[1], 0.25)
          for a in null_alignments(100, rng_9a)]
    rows.append({"aligned to": label, "modulation": chamber.modulation(c, 0.25), "null 95th pct": np.percentile(nm, 95)})
print("\nVariants, 250 ms bins:")
print(pd.DataFrame(rows).round(3).to_string(index=False))
rate = counts / 0.1
for m, lab in ((big, "large"), (~big, "small")):
    r = rate[m].mean(0)
    print(f"{lab}: {m.sum()} trials; mean rate before V (-1 to 0 s) {rate[m][:, centers < 0].mean():.2f} spikes/s, "
          f"0 to 0.5 s {rate[m][:, (centers > 0) & (centers < 0.5)].mean():.2f}, peak {r.max():.2f} at {centers[r.argmax()]:.2f} s")
dip = rate[:, (centers > 0) & (centers < 0.2)].mean()
print(f"All successful trials: mean rate -1 to 0 s {rate[:, centers < 0].mean():.2f}, 0 to 0.2 s {dip:.2f}, 0.3 to 0.5 s "
      f"{rate[:, (centers > 0.3) & (centers < 0.5)].mean():.2f} spikes/s")
post = rate[:, (centers > 0) & (centers < 1)].mean(1)
print(f"Large against small, mean rate 0 to 1 s after V: {post[big].mean():.2f} against {post[~big].mean():.2f} spikes/s, "
      f"Mann-Whitney p = {stats.mannwhitneyu(post[big], post[~big]).pvalue:.3g} (descriptive)")
fig, axs = plt.subplots(2, 1, figsize=(7, 6), sharex=True)
order = np.r_[np.where(big)[0], np.where(~big)[0]]
for row, i in enumerate(order):
    lo, hi = np.searchsorted(spk + OFFSET, [V[i] - 1, V[i] + 3])
    axs[0].plot((spk + OFFSET)[lo:hi] - V[i], np.full(hi - lo, row), "|", ms=1, color="tab:red" if big[i] else "tab:blue")
axs[0].set(ylabel="trial (large, then small)")
for m, lab, col in ((big, "large", "tab:red"), (~big, "small", "tab:blue")):
    axs[1].plot(centers, rate[m].mean(0), color=col, label=lab)
axs[1].axvline(0, color="k", lw=0.5)
axs[1].set(xlabel="time from V (s)", ylabel="spikes/s")
axs[1].legend()
print("PSTH range:", round(min(rate[big].mean(0).min(), rate[~big].mean(0).min()), 2), "to", round(max(rate[big].mean(0).max(), rate[~big].mean(0).max()), 2), "spikes/s")

**Result: reproduced, after the spike times are moved to the trial clock.** With the offset measured in the
data check, every successful trial has a spike within V - 1 s to V + 3 s, and the bin variance of the PSTH is
0.49 against a 95th percentile of 0.06 for random alignments (p = 0.005 with 200 alignments); the Plexon unit
gives 0.17 against 0.06. The rate falls from 5.63 spikes/s before V to 4.58 in the first 0.2 s after it and
rises to 7.70 at 0.3 to 0.5 s, peaking at 0.45 s for large trials (8.5 spikes/s) and 0.35 s for small (8.0). Large and small trials do not differ over 0 to 1 s (5.69 against 5.66 spikes/s,
p = 0.90). With the spike times as stored, only 42% of trials have a spike in the window and the variance is
0.01 against 0.03, so the claim fails on the file as published.

## fig9bc-unit-waveform-isi

> The spikes at site c5d have a consistent waveform (mean and standard deviation) and an interspike
> interval distribution (1 ms bins) consistent with a single unit.

Criterion: fewer than 1% of ISIs below 1 ms, and the largest absolute value of the mean waveform more than
twice the SD across spikes at that sample.

In [ ]:
wf_all = chamber.thresholded_waveforms(f, list(thr).index(main_unit))
isi = np.diff(spk)
mean_wf, sd_wf = wf_all.mean(0), wf_all.std(0)
k = int(np.argmax(np.abs(mean_wf)))
rows = []
for label, times, w in ((f"thresholded unit {main_unit} (main)", spk, wf_all), ("Plexon unit", plx_spk, None)):
    i = np.diff(times)
    row = {"spikes": label, "n": len(times), "n ISI < 1 ms": int(np.sum(i < 0.001)), "frac ISI < 1 ms": np.mean(i < 0.001),
           "n ISI < 2 ms": int(np.sum(i < 0.002)), "frac ISI < 2 ms": np.mean(i < 0.002), "median ISI ms": np.median(i) * 1000}
    if w is not None:
        row.update({"mean wf extreme (uV)": mean_wf[k], "SD there (uV)": sd_wf[k], "ratio": abs(mean_wf[k]) / sd_wf[k]})
    row["shortest ISI ms"] = i.min() * 1000
    rows.append(row)
print(pd.DataFrame(rows).round(6).to_string(index=False))
in_trials = np.zeros(len(spk), bool)
for a in V:
    lo, hi = np.searchsorted(spk + OFFSET, [a - 1, a + 3])
    in_trials[lo:hi] = True
print(f"ISI < 1 ms among spikes within V-1 to V+3 s of successful trials: {np.mean(np.diff(spk[in_trials]) < 0.001):.4f} "
      "(pairs across trial windows included)")
fig, axs = plt.subplots(1, 2, figsize=(9, 3))
t_wf = np.arange(wf_all.shape[1]) / 32.0
axs[0].plot(t_wf, mean_wf, color="k")
axs[0].fill_between(t_wf, mean_wf - sd_wf, mean_wf + sd_wf, color="0.7")
axs[0].set(xlabel="ms", ylabel="uV")
axs[1].hist(isi[isi < 0.1] * 1000, bins=np.arange(0, 101, 1), color="0.4")
axs[1].set(xlabel="ISI (ms), 1 ms bins", ylabel="count")
print(f"Waveform plot range {np.min(mean_wf - sd_wf):.1f} to {np.max(mean_wf + sd_wf):.1f} uV; "
      f"ISI histogram shows {np.mean(isi < 0.1):.1%} of ISIs (those under 100 ms)")

**Result: reproduced.** No ISI is shorter than 1 ms, and the mean waveform's trough (-56.4 µV) is 4.79 times
the SD across spikes at that sample. The ISI part says little: the shortest ISI is 1.56 ms, about the 48-sample
waveform length, so the detection itself excludes shorter intervals (9 ISIs, 0.009%, are under 2 ms). The
Plexon unit has 1 ISI under 1 ms and 39 (0.04%) under 2 ms.

## fig9d-beta-suppression-large

> Trial-averaged beta-band LFP power at c7b referenced to c5c is suppressed more in the brief window after
> the value cue for large than for small reward cues.

Criterion: per successful trial, log10 of mean 13-30 Hz power over 0 to 0.5 s after V divided by the 0.5 s
before V; large lower than small, two-sided Mann-Whitney p < 0.05.

In [ ]:
def beta_test(band, post, base, label):
    t_ax, pw = chamber.band_power_around(lfp_t, lfp_v, V, band=band)
    ratio = np.log10(chamber.window_mean(t_ax, pw, *post) / chamber.window_mean(t_ax, pw, *base))
    a, b = ratio[big], ratio[~big]
    dropped = (int(np.isnan(a).sum()), int(np.isnan(b).sum()))
    a, b = a[~np.isnan(a)], b[~np.isnan(b)]
    p = stats.mannwhitneyu(a, b, alternative="two-sided").pvalue
    w_a, w_b = stats.wilcoxon(a).pvalue, stats.wilcoxon(b).pvalue
    return {"variant": label, "n_large": len(a), "n_small": len(b), "dropped (l, s)": dropped,
            "median_large": np.median(a), "median_small": np.median(b), "diff_medians": np.median(a) - np.median(b),
            "mann_whitney_p": p, "p large != 0": w_a, "p small != 0": w_b}, t_ax, pw
rows = []
r, t_beta, pw_beta = beta_test((13, 30), (0, 0.5), (-0.5, 0), "13-30 Hz, 0-0.5 s vs 0.5 s before (main)")
rows.append(r)
for band, post, base, label in [((13, 30), (0, 0.3), (-0.5, 0), "post 0-0.3 s"), ((13, 30), (0, 1.0), (-0.5, 0), "post 0-1 s"),
                                ((13, 30), (0, 0.5), (-1.0, 0), "1 s baseline"), ((15, 30), (0, 0.5), (-0.5, 0), "15-30 Hz"),
                                ((12, 25), (0, 0.5), (-0.5, 0), "12-25 Hz")]:
    rows.append(beta_test(band, post, base, label)[0])
print(pd.DataFrame(rows).round(4).to_string(index=False))
raw_post = chamber.window_mean(t_beta, pw_beta, 0, 0.5)
ok = ~np.isnan(raw_post)
print(f"Variant without baseline: mean beta power 0-0.5 s, large {np.median(raw_post[big & ok]):.3e} against small {np.median(raw_post[~big & ok]):.3e} V^2 (medians), "
      f"Mann-Whitney p = {stats.mannwhitneyu(raw_post[big & ok], raw_post[~big & ok]).pvalue:.3g}")
for name, m in (("first half of session", half), ("second half of session", ~half)):
    t_ax, pw = t_beta, pw_beta[m]
    ratio = np.log10(chamber.window_mean(t_ax, pw, 0, 0.5) / chamber.window_mean(t_ax, pw, -0.5, 0))
    a, b = ratio[big[m]], ratio[~big[m]]
    a, b = a[~np.isnan(a)], b[~np.isnan(b)]
    print(f"{name}: medians large {np.median(a):.4f}, small {np.median(b):.4f}, p = {stats.mannwhitneyu(a, b).pvalue:.3g}")
fig, ax = plt.subplots(figsize=(7, 3))
for m, lab, col in ((big, "large", "tab:red"), (~big, "small", "tab:blue")):
    base = chamber.window_mean(t_beta, pw_beta[m], -0.5, 0)
    norm = pw_beta[m] / base[:, None]
    ax.plot(t_beta, np.nanmean(norm, 0), color=col, label=lab)
ax.axvline(0, color="k", lw=0.5)
ax.set(xlabel="time from V (s)", ylabel="beta power / 0.5 s before V", xlim=(-1.5, 3))
ax.legend()
lines = [np.nanmean(pw_beta[m] / chamber.window_mean(t_beta, pw_beta[m], -0.5, 0)[:, None], 0) for m in (big, ~big)]
sel = (t_beta >= -1.5) & (t_beta <= 3)
print("Plotted range:", round(min(l[sel].min() for l in lines), 3), "to", round(max(l[sel].max() for l in lines), 3))

**Result: partially reproduced.** Beta power (13 to 30 Hz) falls after V in both conditions (median log10
ratio -0.158 for large and -0.125 for small, each p < 0.001 against 0), and the fall is larger for large
reward cues, but not significantly (difference of medians -0.033, Mann-Whitney p = 0.108). No variant reaches
p < 0.05: 15 to 30 Hz gives p = 0.063, a 1 s baseline p = 0.108, and the 0 to 0.3 s and 0 to 1 s windows show
no difference (p = 0.59 and 0.84).

## Summary

A failure to reproduce here may be a failure of this attempt, not of the paper: see the reasoning and the
differences from the paper's method.

| Claim | Outcome |
|---|---|
| fig8a-single-trial-dopamine | reproduced |
| fig8b-dopamine-large-vs-small | reproduced |
| fig9a-spikes-task-aligned | reproduced (after correcting the spike clock) |
| fig9bc-unit-waveform-isi | reproduced |
| fig9d-beta-suppression-large | partially reproduced |
| recordings-two-monkeys | blocked: data missing |
| scfe-functional-yield | blocked: data missing |
| scfe-background-yield | blocked: data missing |
| mcfe-functional-yield | blocked: data missing |
| dopamine-signal-yield | blocked: data missing |
| asepsis-cultures | blocked: data missing |
| fig6-drainage-volume | blocked: data missing |
| fig7-bone-growth-margins | blocked: data missing |

**How the analysis differs from the paper's.** The paper gives no analysis methods for Figures 8 and 9 beyond
the alignment to V and the background subtraction at V. The windows, the beta band (13 to 30 Hz), the power
measure, the tests and the ISI and waveform criteria are choices made here and fixed in the claims file. Dopamine
is referenced at V by subtracting the PCA estimate at V, which equals background subtraction at V when the PCA
model is linear. The spike times are moved by the offset measured in the data check (11812.11 s). Trial 1411,
which the paper's large reward list leaves out, is kept.

**What was found about the dandiset.**

- The raw ephys and the spike times are not on the clock of the trials, events, LFP and FSCV table. The raw
  ephys is stored from 0 s with no timestamps; locating it in the stored LFP puts it at 11812.11 s, with a
  drift of about 4 ms per hour. Without this correction the spike claim fails.
- The absolute times disagree: on the trial clock the raw samples end at 17:12:20, two hours after the
  recording-closed time in the Neuralynx header (15:12:24).
- The stored spike times are about 7.1 ms after the spike troughs in raw c5d.
- The "beta-band LFP" is broadband, its values are already volts, and it carries the raw int16 conversion
  factor. The thresholded waveforms are stated in volts but hold µV; the raw FSCV current is stated in amperes
  but holds nA. The trial-aligned FSCV columns state no units.
- The subject description, session and experiment descriptions and one keyword describe the lab's
  micro-invasive probe study, not this paper.
- The event table repeats each event once per overlapping 60 s trial window (up to five copies, unsorted).
  The trial tags carry no trial type; reward size and outcome come from the events, and they reproduce the
  paper's trial lists.
- The Plexon unit's spike times are not sorted.
- The raw FSCV minute files are joined without their gaps (328 min against 302 min of trials), and the
  trial-aligned oxidation current cannot be found in them, so the raw FSCV's clock could not be established.
- Raw c5c sits at the input range in about 1.7% of samples, and the raw signals carry a large 10 Hz artefact
  (the FSCV scan rate).
- The reward size is said to depend on whether the target is at the top or bottom of the screen; the events
  code left and right conditions, which do not predict reward size.
- Only monkey T and one session are deposited; the data behind the yield, culture, drainage and bone-growth
  results are not.

In [ ]:
print(f"Notebook ran in {(time.time() - started) / 60:.1f} min")